In [66]:
using Symbolics
using Latexify
using PrettyTables
using SymbolicUtils
using SymbolicUtils.Rewriters

# Dedução das equações de movimento e controle de um Pêndulo Invertido

<div style="text-align: center;">
  <img src="Imagens/PênduloInvertido/penduloInvertido.png" alt="alt text" width="20%">
</div>

Este documento detalha a dedução do modelo matemático de um pêndulo invertido
montado sobre um carro móvel, considerando atrito em ambas as juntas e que a massa do pêndulo está concentrada na ponta do mesmo. Além disso, o método
utilizado para a dedução das equações de movimento é o formalismo de Euler-Lagrange.

## Parâmetros do Sistema

* $M$: Massa do carro.
* $m$: Massa do pêndulo.
* $L$: Comprimento da haste do pêndulo (distância até o centro de massa).
* $g$: Aceleração da gravidade.
* $b$: Coeficiente de atrito viscoso entre o carro e o trilho.
* $d$: Coeficiente de atrito viscoso no pivô do pêndulo.
* $F$: Força de controle aplicada horizontalmente no carro.



## Coordenadas Generalizadas

O sistema possui *dois graus de liberdade*, representados pelo vetor $q = [x, \theta]^T$ , onde
$x$ é a posição horizontal do carro e $\theta$ é o ângulo do pêndulo em relação à vertical
(com $\theta = 0$ indicando a posição perfeitamente vertical para cima e $\theta = \pi$ indicando
a posição perfeitamente vertical para baixo).

<div style="text-align: center;">
  <img src="Imagens\PênduloInvertido\penduloInvertido_GrausLiberdade.PNG" alt="alt text" width="20%">
  <img src="Imagens\PênduloInvertido\penduloInvertido_estadosEquilibrio.png" alt="alt text" width="20%">
</div>

## Cinemática do Sistema

<div style="text-align: center;">
  <img src="Imagens/PênduloInvertido/penduloInvertido_posicaoMassa.png" alt="alt text" width="20%">
</div>

A posição do centro de massa do carro é simplesmente $(x, 0)$. A posição do centro
de massa do pêndulo $(x_m, y_m)$ é dada pelas relações trigonométricas:

$
x_m = x + L  \sin θ \\
y_m = L  \cos θ 
$

As velocidades são obtidas derivando as posições em relação ao tempo:

$
\dot{x}_m = \dot{x} + L  \dot{\theta}  \cos\theta \\
\dot{y}_m = -L  \dot{\theta}  \sin\theta
$



In [74]:
@variables t L x(t) θ(t) 

D = Differential(t)

x_m = x + L*sin(θ)
y_m = -L*cos(θ)

dx_m = expand_derivatives(D(x_m)); dy_m = expand_derivatives(D(y_m));

display(x_m); display(y_m); display(dx_m); display(dy_m);

x(t) + L*sin(θ(t))

-L*cos(θ(t))

Differential(t, 1)(x(t)) + L*cos(θ(t))*Differential(t, 1)(θ(t))

L*Differential(t, 1)(θ(t))*sin(θ(t))

## Energias do Sistema e o Lagrangiano

O formalismo de Euler-Lagrange exige que seja obtido as energias cinéticas e potênciais do sistema, sendo que a diferença entre elas define uma função fundamental chamada Lagrangiana ($\mathcal{L}$)

$$
\mathcal{L} = T - V
$$

Diferente das Leis de Newton, onde precisamos desenhar vetores de força para cada componente, o método de Lagrange utiliza coordenadas generalizadas ($q_i$). Essas coordenadas são as variáveis mínimas necessárias para descrever a posição do sistema (como o ângulo $\theta$ de um pêndulo ou a posição $x$ de um carrinho). Essa teoria afirma que a partir dessa equação: 

$$
\frac{d}{dt} \left( \frac{\partial L}{\partial \dot{q}_i} \right) - \frac{\partial L}{\partial q_i} = 0
$$


### Energia Cinética

Ao resolver essa equação para cada coordenada, obtemos as equações de movimento do sistema. Assim a energia cinética do sistema analisado é:

$$
T = \frac{1}{2} M \dot{x}^2 + \frac{1}{2} m \left( \dot{x}_m^2 + \dot{y}_m^2 \right)
$$

*Aqui é interessante notar que como o sistema tem dois corpos conectados é necessária a soma das energias cinética de ambos!*

Substituindo as velocidades e utilizando a identidade sin2 θ + cos2 θ = 1:

$$
T = \frac{1}{2} M \dot{x}^2 + \frac{1}{2} m \left( (\dot{x} + L \dot{\theta} \cos\theta)^2 + (-L \dot{\theta} \sin\theta)^2 \right) \nonumber \\
T = \frac{1}{2} (M + m) \dot{x}^2 + m L \dot{x} \dot{\theta} \cos\theta + \frac{1}{2} m L^2 \dot{\theta}^2
$$

In [ ]:
@variables M m

T = 0.5*M*dx_m^2 + 0.5*m*(dx_m^2 + dy_m^2)

0.5((Differential(t, 1)(x(t)) + L*cos(θ(t))*Differential(t, 1)(θ(t)))^2)*M + 0.5((Differential(t, 1)(x(t)) + L*cos(θ(t))*Differential(t, 1)(θ(t)))^2 + (L^2)*(Differential(t, 1)(θ(t))^2)*(sin(θ(t))^2))*m

### Energia Potencial

A energia potencial $V$ do sistema provém apenas da elevação da massa $m$ do pêndulo, uma vez que o carrinho não armazena energia potência devido ao seu movimento:
$$
V = m g L \cos\theta
$$

In [72]:
@variables g

V = m*g*L

L*g*m

### O Lagrangiano

O Lagrangiano $\mathcal{L} = T - V$ do sistema é:
$$
    \mathcal{L} = \frac{1}{2} (M + m) \dot{x}^2 + m L \dot{x} \dot{\theta} \cos\theta + \frac{1}{2} m L^2 \dot{\theta}^2 - m g L \cos\theta
$$

In [73]:
L = T - V

-L*g*m + 0.5((Differential(t, 1)(x(t)) + L*cos(θ(t))*Differential(t, 1)(θ(t)))^2)*M + 0.5((Differential(t, 1)(x(t)) + L*cos(θ(t))*Differential(t, 1)(θ(t)))^2 + (L^2)*(Differential(t, 1)(θ(t))^2)*(sin(θ(t))^2))*m